# 14 — Role-Playing Dimension + Degenerate Dimension — Spark SQL

## Role-Playing Dimension
`DimDate` aparece 3x no `FactOrderFulfillment`: OrderDate, RequiredDate, ShippedDate.
Views SQL com nomes semânticos.

## Degenerate Dimension
`OrderID` em `FactSales` — chave de negócio sem tabela dimensão própria.

In [1]:
import sys
import os
sys.path.insert(0, os.getcwd())
from utils import get_spark, register_catalog, WAREHOUSE_DIR

spark = get_spark("NorthwindDW SQL - 14 Role Playing")
print("Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/29 01:03:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.0


In [2]:
register_catalog(spark)

Catálogo registrado: {'bronze': 11, 'silver': 0, 'gold': 15}


In [3]:
spark.sql("CREATE OR REPLACE VIEW gold.v_order_date    AS SELECT * FROM gold.DimDate")
spark.sql("CREATE OR REPLACE VIEW gold.v_required_date AS SELECT * FROM gold.DimDate")
spark.sql("CREATE OR REPLACE VIEW gold.v_shipped_date  AS SELECT * FROM gold.DimDate")

expected = spark.sql("SELECT COUNT(*) AS n FROM gold.DimDate").collect()[0]["n"]
for v in ["gold.v_order_date", "gold.v_required_date", "gold.v_shipped_date"]:
    n = spark.sql(f"SELECT COUNT(*) AS n FROM {v}").collect()[0]["n"]
    assert n == expected, f"{v} diverge de DimDate!"
print("Role-playing views criadas OK")

Role-playing views criadas OK


In [4]:
spark.sql("""
    SELECT od.Year, od.Quarter,
           COUNT(DISTINCT f.OrderID) AS TotalPedidos,
           ROUND(AVG(CAST(f.DaysToShip AS DOUBLE)), 1) AS MediaDias,
           SUM(CASE WHEN f.IsLate THEN 1 ELSE 0 END) AS Atrasados
    FROM gold.FactOrderFulfillment f
    JOIN gold.v_order_date    od ON od.DateKey = f.OrderDateKey
    LEFT JOIN gold.v_shipped_date  sd ON sd.DateKey = f.ShippedDateKey
    LEFT JOIN gold.v_required_date rd ON rd.DateKey = f.RequiredDateKey
    GROUP BY od.Year, od.Quarter ORDER BY od.Year, od.Quarter
""").show()

+----+-------+------------+---------+---------+
|Year|Quarter|TotalPedidos|MediaDias|Atrasados|
+----+-------+------------+---------+---------+
|1996|      3|          70|      8.9|        5|
|1996|      4|          82|      7.5|        2|
|1997|      1|          92|      9.2|        5|
|1997|      2|          93|      9.0|        4|
|1997|      3|         103|      8.2|        5|
|1997|      4|         120|      9.2|        8|
|1998|      1|         182|      8.6|        8|
|1998|      2|          88|      6.4|        0|
+----+-------+------------+---------+---------+



In [5]:
spark.sql("""
    SELECT fs.OrderID,
           dc.CompanyName, dp.ProductName, fs.Quantity, fs.NetRevenue
    FROM gold.FactSales fs
    JOIN gold.DimCustomer dc ON dc.CustomerSK = fs.CustomerSK AND dc.IsCurrent
    JOIN gold.DimProduct  dp ON dp.ProductSK  = fs.ProductSK  AND dp.IsCurrent
    WHERE fs.OrderID = 10248
""").show()

+-------+--------------------+--------------------+--------+----------+
|OrderID|         CompanyName|         ProductName|Quantity|NetRevenue|
+-------+--------------------+--------------------+--------+----------+
|  10248|Vins et alcools C...|      Queso Cabrales|      12|     168.0|
|  10248|Vins et alcools C...|Mozzarella di Gio...|       5|     174.0|
|  10248|Vins et alcools C...|Singaporean Hokki...|      10|      98.0|
+-------+--------------------+--------------------+--------+----------+

